In [1]:
import pandas as pd

In [2]:
ICD_CODES = ["C61", "C34", "C50", "C25", "C22", "C18", "C64", "C80", "C83", "C91"]

In [5]:
MODEL = "GPT_OSS_MODEL_SIZE.SMALL_Low"
TRY = 0
scores = pd.read_csv(f"scores/scores_{MODEL}_{TRY}.csv").rename(columns={"Unnamed: 0": "icd10_category"})
scores = scores[["icd10_category", *ICD_CODES]]

In [6]:
scores.head()

,icd10_category,C61,C34,C50,C25,C22,C18,C64,C80,C83,C91
0,A00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,A01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,A02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,A04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
scores = scores.melt(
    id_vars=["icd10_category"], 
    var_name="icd10_category_1", 
    value_name="score"
).rename(columns={"icd10_category": "icd10_category_2"})

In [11]:
scores.head()

,icd10_category_2,icd10_category_1,score
0,A00,C61,0.0
1,A01,C61,0.0
2,A02,C61,0.0
3,A03,C61,0.0
4,A04,C61,0.0


In [12]:
scores["score"].value_counts()

score
0.0    19867
0.5      525
1.0       48
Name: count, dtype: int64

In [7]:
MODEL = "GPT_OSS_MODEL_SIZE.SMALL_Low"

TRY = 0
scores_mcq = pd.read_csv(f"scores_mcq/scores_{MODEL}_{','.join(ICD_CODES)}_{TRY}.csv").drop("Unnamed: 0", axis=1, errors="ignore")

In [8]:
scores_mcq.head()

,icd10_category_1,icd10_category_2,score
0,C61,K76,0.0
1,C61,R18,0.0
2,C61,K74,0.0
3,C61,B19,0.0
4,C61,J44,0.0


In [9]:
scores_mcq['score'].value_counts()

score
0.0    16164
0.5      747
1.0       49
Name: count, dtype: int64

In [13]:
all_scores = pd.merge(
    scores,
    scores_mcq,
    how="inner",
    left_on=["icd10_category_1", "icd10_category_2"],
    right_on=["icd10_category_1", "icd10_category_2"],
    suffixes=("", "_mcq")
)

In [14]:
all_scores.shape

(16960, 4)

In [15]:
all_scores.head(2)

,icd10_category_2,icd10_category_1,score,score_mcq
0,A01,C61,0.0,0.0
1,A02,C61,0.0,0.0


In [16]:
all_scores["is_equal"] = (all_scores["score"] == all_scores["score_mcq"]).astype(int)

In [17]:
all_scores["is_equal"].sum()/len(all_scores)

0.9297759433962264

In [18]:
all_scores_no_zero = all_scores[(all_scores["score"] != 0) | (all_scores["score_mcq"] != 0)]
all_scores_no_zero["is_equal"].sum()/len(all_scores_no_zero)

0.06367924528301887

In [19]:
import sklearn
import sklearn.metrics 

In [21]:
for hard_metric in ["accuracy", "f1", "precision", "recall"]:
    print(
        hard_metric, ":", 
        round(getattr(sklearn.metrics, hard_metric+"_score")(
            all_scores['score_mcq'] >= 0.5, 
            all_scores['score'] >= 0.5
        ), 4)
    )

for soft_metric in ["roc_auc", "average_precision"]:
    print(
        soft_metric, ":", 
        round(getattr(sklearn.metrics, soft_metric+"_score")(
            all_scores['score_mcq'] >= 0.5, 
            all_scores['score']
        ), 4)
    )

accuracy : 0.9305
f1 : 0.1376
precision : 0.1649
recall : 0.1181
roc_auc : 0.5443
average_precision : 0.0607


In [ ]:
results = []

for icd_code, icd_code_df in all_scores.groupby("icd10_category_1"):
    idc_code_result = {"icd_code": icd_code}
    for hard_metric in ["accuracy", "f1", "precision", "recall"]:
        idc_code_result[hard_metric] = round(
            getattr(sklearn.metrics, hard_metric+"_score")(
                icd_code_df['score_mcq'] >= 0.5, 
                icd_code_df['score'] >= 0.5
            ), 
            4
        )

    for soft_metric in ["roc_auc", "average_precision"]:
        idc_code_result[soft_metric] = round(
            getattr(sklearn.metrics, soft_metric+"_score")(
                icd_code_df['score_mcq'] >= 0.5, 
                icd_code_df['score']
            ), 
            4
        )

    icd_code_df_no_zero = icd_code_df[(icd_code_df["score"] != 0) | (icd_code_df["score_mcq"] != 0)]
    icd_code_df_no_zero["is_equal"] = (icd_code_df_no_zero["score"] == icd_code_df_no_zero["score_mcq"]).astype(int)
    idc_code_result["intersection_of_!=0_elements"] = round(icd_code_df_no_zero["is_equal"].sum()/len(icd_code_df_no_zero), 4)

    results.append(idc_code_result)

results_df = pd.DataFrame(results)
results_df

/tmp/ipykernel_224960/3687899070.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  icd_code_df_no_zero["is_equal"] = (icd_code_df_no_zero["score"] == icd_code_df_no_zero["score_mcq"]).astype(int)
/tmp/ipykernel_224960/3687899070.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  icd_code_df_no_zero["is_equal"] = (icd_code_df_no_zero["score"] == icd_code_df_no_zero["score_mcq"]).astype(int)
/tmp/ipykernel_224960/3687899070.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,icd_code,accuracy,f1,precision,recall,roc_auc,average_precision,intersection_of_!=0_elements
0,C18,0.9404,0.1789,0.2037,0.1594,0.5662,0.0667,0.080357
1,C22,0.9334,0.1630,0.2292,0.1264,0.5517,0.0735,0.080645
2,C25,0.9239,0.1103,0.1702,0.0816,0.5286,0.0673,0.036496
3,C34,0.9204,0.1916,0.2581,0.1524,0.5617,0.0917,0.086093
4,C50,0.9428,0.1849,0.2245,0.1571,0.5672,0.0827,0.101852
5,C61,0.9469,0.1509,0.1538,0.1481,0.5607,0.0499,0.071429
6,C64,0.9570,0.2151,0.2439,0.1923,0.5867,0.0717,0.108434
7,C80,0.8897,0.0508,0.0340,0.1000,0.5082,0.0336,0.020833
8,C83,0.9375,0.1311,0.2500,0.0889,0.5369,0.0706,0.070175
9,C91,0.9133,0.0755,0.1579,0.0496,0.5146,0.0756,0.032680


In [28]:
results_df.drop(columns=["icd_code"]).mean(axis=0).round(4)

accuracy                        0.9305
f1                              0.1452
precision                       0.1925
recall                          0.1256
roc_auc                         0.5482
average_precision               0.0683
intersection_of_!=0_elements    0.0689
dtype: float64